In [109]:
import numpy as np
import pandas as pd
np.random.seed(42)
TOTAL_COUNT_RAND = 10000
NULL_RATIO_RAND = 0.05
NULL_COUNT_RAND = int(TOTAL_COUNT_RAND * NULL_RATIO_RAND)
VALID_COUNT_RAND = TOTAL_COUNT_RAND - NULL_COUNT_RAND

## Features
- Tenure
- Tech Support
- Contract
- Monthly Charges
## Labels
- Churn

# Random Uncorrelational Dataset


In [110]:
# Random Uncorrelational Dataset
tenure_rand = np.random.randint(1,73,size=VALID_COUNT_RAND)
monthly_charges_rand = (120.00 - 15.00) * np.random.ranf(size=VALID_COUNT_RAND) + 15.00
plans = ["Month-to-month", "One year", "Two year"]
contract_rand = np.random.choice(plans, size=TOTAL_COUNT_RAND, replace=True)
bins = ["Yes", "No"]
tech_support_rand = np.random.choice(bins,size=TOTAL_COUNT_RAND, replace=True)
churn_rand = np.random.choice(bins,size=TOTAL_COUNT_RAND, replace=True)

In [111]:
null_arr = np.array([np.nan] * NULL_COUNT_RAND)
tenure_rand = np.random.permutation(np.append(tenure_rand, null_arr))
monthly_charges_rand = np.random.permutation(np.append(monthly_charges_rand, null_arr))

In [112]:
customer_id = ['CUST-' + str(x).zfill(4) for x in np.arange(TOTAL_COUNT_RAND)]

# Correlational Data Generation


In [113]:
TOTAL_COUNT= 10000
NULL_RATIO = 0.05
NULL_COUNT = int(TOTAL_COUNT * NULL_RATIO)
VALID_COUNT = TOTAL_COUNT - NULL_COUNT

In [114]:
# Random Tenure Array, Our Anchor
tenure = np.random.randint(1,73, size=TOTAL_COUNT)

# Random Tech Support
tech_support = np.random.choice(bins,size=TOTAL_COUNT, replace=True)

# Customer Id
customer_id = ['CUST-' + str(x).zfill(4) for x in np.arange(TOTAL_COUNT)]


In [115]:
plans = ["Month-to-month", "One year", "Two year"]

long_term = np.random.choice(plans, size=TOTAL_COUNT, p=[0.1,0.4,0.5])
short_term = np.random.choice(plans, size=TOTAL_COUNT, p=[0.8,0.2,0.0])

# Weighted Contracts
contract = np.where(tenure > 24, long_term, short_term)

In [116]:
base_charge = 70.00
mm_charge = 25.00 # $25 Monthly
yy_charge = 20.00 # $240 Yearly
yyyy_charge = 15.00 # 
ts_charge = 25.00

conditions = [
    contract == "Month-to-month",
    contract == "One year",
    contract == "Two year"
]
choices = [
    mm_charge,
    yy_charge,
    yyyy_charge
]
monthly_charges = np.full(TOTAL_COUNT, base_charge)
monthly_charges += np.select(conditions, choices, default=0)
monthly_charges += np.where(tech_support == "Yes", ts_charge, 0)

In [117]:
from sklearn.preprocessing import MinMaxScaler
# Weights
W1 = 1
W2 = 1
W3 = 0.5
W4 = 3

tenure_w = (tenure / max(tenure)) * W1
monthly_charges_w = (monthly_charges / max(monthly_charges)) * W2
tech_support_w = np.where(tech_support == "Yes", 1, 0) * W3
contract_w = np.select(conditions, choicelist=[1,0,-1], default=0) * W4


churn_risk = np.zeros(TOTAL_COUNT) - tenure_w - tech_support_w + contract_w + monthly_charges_w
n_scaler = MinMaxScaler()
churn_prob = n_scaler.fit_transform(churn_risk.reshape(-1, 1)).flatten()

random_roll = np.random.rand(TOTAL_COUNT)
churn = np.where(random_roll < churn_prob, "Yes", "No")

## Fair Coin Toss
- churn = np.random.choice(["Yes", "No"])

## Weighted Coin Toss
- random_roll = np.random.rand(TOTAL_COUNT)
- churn = np.where(random_roll < churn_prob, "Yes", "No")

In [118]:
# Injecting NaNs
np.random.seed(42)
rand_idx1 = np.random.choice(TOTAL_COUNT, size=NULL_COUNT, replace=False)
rand_idx2 = np.random.choice(TOTAL_COUNT, size=NULL_COUNT, replace=False)
tenure = tenure.astype(dtype=float)
tenure[rand_idx1] = np.nan
monthly_charges[rand_idx2] = np.nan

df = pd.DataFrame({
    "customer_id" : customer_id,
    "tenure" : tenure, 
    "monthly_charges" : monthly_charges,
    "contract" : contract,
    "tech_support" : tech_support, 
    "churn" : churn
    })

In [119]:
df.head()

,customer_id,tenure,monthly_charges,contract,tech_support,churn
0,CUST-0000,70.0,90.0,One year,No,No
1,CUST-0001,24.0,95.0,Month-to-month,No,Yes
2,CUST-0002,60.0,90.0,One year,No,No
3,CUST-0003,NaN,120.0,Month-to-month,Yes,Yes
4,CUST-0004,47.0,90.0,One year,No,No


In [120]:
df.replace({np.nan : None}, inplace=True)
df.head()

,customer_id,tenure,monthly_charges,contract,tech_support,churn
0,CUST-0000,70.0,90.0,One year,No,No
1,CUST-0001,24.0,95.0,Month-to-month,No,Yes
2,CUST-0002,60.0,90.0,One year,No,No
3,CUST-0003,None,120.0,Month-to-month,Yes,Yes
4,CUST-0004,47.0,90.0,One year,No,No


In [121]:
import sys
import os

root_dir = os.path.abspath('..')

if root_dir not in sys.path:
    sys.path.append(root_dir)

In [ ]:
from database.db_config import db_connect
from psycopg2 import extras

insert_query = """
INSERT INTO customers (customer_id, tenure, monthly_charges, contract, tech_support, churn)
VALUES %s
"""
data_tuples = [tuple(x) for x in df.to_numpy()]

conn = db_connect()
cur = conn.cursor()

try:
    cur.execute("TRUNCATE TABLE customers;")
    print("Table Cleared")
    extras.execute_values(cur, insert_query, data_tuples, page_size=1000)
    print("Data Inserted Successfuly")
    conn.commit()
    cur.close()
except Exception as e:
    print(f"Data Not Inserted. Error : {e}")
    conn.rollback()
finally:
    if conn:
        conn.close()
        



Database connected successfully
